# GroupBy Aggregation & Segment Insights

This notebook shows how to turn a flat transaction table into segment-level business insights using groupby, multi-level aggregation, pivot tables, ranking, and action-oriented interpretation.

## Setup Note

The notebook uses a small self-contained demo dataset so the workflow is easy to follow. If you have a real dataset, replace the demo dataframe with your own `df` containing at least `customer_type`, `product`, `revenue`, `churn`, and `customer_id`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'customer_type': ['Enterprise', 'Enterprise', 'SMB', 'SMB', 'SMB', 'Startup', 'Startup', 'Startup', 'Enterprise', 'SMB', 'Startup', 'Enterprise'],
    'product': ['A', 'B', 'A', 'B', 'A', 'A', 'B', 'C', 'C', 'C', 'B', 'A'],
    'revenue': [120000, 98000, 15000, 18000, 22000, 6000, 5000, 7000, 105000, 25000, 8000, 115000],
    'churn': [0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
})

df.head()

## GroupBy Fundamentals

Groupby follows the split-apply-combine pattern. Split the data into groups, apply an aggregation or custom calculation to each group, then combine the results into a summary table. The three most common methods are `agg`, `transform`, and `apply`.

In [ ]:
# .agg() - Combine results into single value per group
agg_example = df.groupby('customer_type')['churn'].agg(['sum', 'count', 'mean'])
print('agg() example:
', agg_example, '
')

# .transform() - Return result same shape as original (broadcast per row)
df['churn_rate_by_type'] = df.groupby('customer_type')['churn'].transform('mean')
print('transform() example:
', df[['customer_type', 'churn', 'churn_rate_by_type']].head(), '
')

# .apply() - Custom function per group
top_revenue_sum = df.groupby('customer_type')['revenue'].apply(lambda x: x.nlargest(3).sum())
print('apply() example:
', top_revenue_sum)

## Task 1: Segment-Level Aggregation

The first step is to group by customer type and compute the churn rate, total revenue, and customer count for each segment. This replaces a misleading dataset-wide average with segment-specific business insight.

In [ ]:
segment_metrics = df.groupby('customer_type').agg({
    'churn': 'mean',
    'revenue': 'sum',
    'customer_id': 'count'
})

segment_metrics.columns = ['churn_rate', 'total_revenue', 'customer_count']
segment_metrics = segment_metrics.sort_values('churn_rate', ascending=False)
print(segment_metrics)

## Task 2: Multi-Dimensional Aggregation

Grouping by both customer type and product exposes where revenue is actually coming from. A two-dimensional summary is often more useful than a single total because it shows which segment-product combinations are healthy or weak.

In [ ]:
segment_product_revenue = df.groupby(['customer_type', 'product'])['revenue'].sum()
print('Multi-level groupby result:
', segment_product_revenue, '
')

print('Unstacked view:
', segment_product_revenue.unstack(fill_value=0), '
')

pivot = pd.pivot_table(
    df,
    values='revenue',
    index='customer_type',
    columns='product',
    aggfunc='sum',
    fill_value=0
)
print('Pivot table:
', pivot)

## Task 3: Rank Segments

Ranking helps identify the strongest and weakest customer groups. In the business scenario, enterprise customers should typically show low churn and high revenue, while SMB and startup segments may need more attention.

In [ ]:
segment_metrics['churn_rank'] = segment_metrics['churn_rate'].rank(method='dense', ascending=False)
segment_metrics['revenue_rank'] = segment_metrics['total_revenue'].rank(method='dense', ascending=False)

print(segment_metrics.sort_values(['churn_rank', 'revenue_rank']))

worst_segments = segment_metrics.sort_values('churn_rate', ascending=False).head(2)
best_segments = segment_metrics.sort_values('churn_rate', ascending=True).head(2)

print('Worst segments:
', worst_segments, '
')
print('Best segments:
', best_segments)

## Task 4: Surface Actionable Insights

Segment summaries should always lead to action. For each segment, connect the metric pattern to a business response instead of just listing numbers.

In [ ]:
insights = []
for customer_type, row in segment_metrics.sort_values('churn_rate', ascending=False).iterrows():
    if customer_type == 'Enterprise':
        action = 'Protect with white-glove support and retention programs.'
    elif customer_type == 'SMB':
        action = 'Reduce friction and improve onboarding to lower churn.'
    else:
        action = 'Improve activation and early-life engagement.'
    insights.append({
        'segment': customer_type,
        'churn_rate': round(row['churn_rate'], 2),
        'total_revenue': int(row['total_revenue']),
        'customer_count': int(row['customer_count']),
        'action': action
    })

print(json.dumps(insights, indent=2))

## Task 5: Visualize Segment Differences

A compact bar chart makes it easy to compare churn rate and revenue across segments. This is often the fastest way to brief a business stakeholder.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segment_metrics['churn_rate'].sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='#c44e52')
axes[0].set_title('Churn Rate by Customer Type')
axes[0].set_ylabel('Churn Rate')

segment_metrics['total_revenue'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='#4c72b0')
axes[1].set_title('Total Revenue by Customer Type')
axes[1].set_ylabel('Revenue')

plt.tight_layout()
plt.savefig('output/segment_insights.png')
plt.show()

## Business Summary

The main lesson is to segment first and summarize second. A single churn rate can hide a healthy enterprise group and a problematic SMB group. Groupby makes the differences visible, and the resulting action should be segment-specific rather than one-size-fits-all.

## Submission

Submit the completed notebook and the GitHub PR link together. If either one is missing, the submission is incomplete.

GitHub PR link: [paste your PR URL here]